# 2.2 SHV 与 HSHV 模式对比

**学习目标**
- 理解同时收发（SHV）和交替收发（HSHV）模式的工作原理
- 比较两种模式在信号采样率和通道一致性方面的差异
- 了解HSHV模式下差分相位的估计方法
- 分析两种模式的适用场景

**参考文献**：Ryzhkov & Zrnic, Chapter 2, pp. 161-195

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

ModuleNotFoundError: No module named 'numpy'

## 1. 理论背景

### SHV 模式（Simultaneous H and V）

同时发射H和V偏振波，同时接收H和V通道的回波信号。

- **优点**：时间采样率高，适合快速变化的降水过程
- **缺点**：发射功率分散到两个通道，灵敏度降低约3dB

### HSHV 模式（Hybrid SHV）

交替发射H偏振和V偏振脉冲，在每个脉冲重复周期内完成一次完整测量。

- **优点**：发射功率集中，灵敏度更高
- **缺点**：时间采样率减半，可能丢失快速变化的特征

In [ ]:
def simulate_shv_hshv(velocity=20.0, wavelength=0.10, prf=1000, n_pulses=64,
                        differential_reflectivity=2.0, differential_phase=30.0):
    """
    模拟 SHV 和 HSHV 模式的信号
    
    differential_reflectivity: ZDR (dB)
    differential_phase: ΦDP (度)
    """
    # 参数转换
    zdr_linear = 10**(differential_reflectivity/10)
    phi_dp_rad = np.radians(differential_phase)
    
    # 多普勒频移
    f_d = 2 * velocity / wavelength
    
    # 时间序列
    t = np.arange(n_pulses) / prf
    
    # ===== SHV 模式: 同时收发 =====
    # H通道和V通道同时发射和接收
    phase_h = 2 * np.pi * f_d * t
    phase_v = phase_h + phi_dp_rad
    
    # 幅度比 (与ZDR相关)
    amp_h = np.sqrt(zdr_linear)
    amp_v = 1.0 / np.sqrt(zdr_linear)
    
    Zh = amp_h * np.exp(1j * phase_h)
    Zv = amp_v * np.exp(1j * phase_v)
    
    # ===== HSHV 模式: 交替收发 =====
    # 偶数脉冲发H收H，奇数脉冲发V收V
    n_h = n_pulses // 2
    n_v = n_pulses - n_h
    
    # H脉冲序列
    t_h = np.arange(n_h) * 2 / prf  # 间隔2个PRT
    phase_h_hshv = 2 * np.pi * f_d * t_h
    Zh_hshv = amp_h * np.exp(1j * phase_h_hshv)
    
    # V脉冲序列
    t_v = np.arange(n_v) * 2 / prf + 1/prf  # 偏移半个PRT
    phase_v_hshv = 2 * np.pi * f_d * t_v + phi_dp_rad
    Zv_hshv = amp_v * np.exp(1j * phase_v_hshv)
    
    return {
        'shv': {'Zh': Zh, 'Zv': Zv, 't': t},
        'hshv': {'Zh': Zh_hshv, 'Zv': Zv_hshv, 't_h': t_h, 't_v': t_v},
        'f_d': f_d, 'phi_dp': phi_dp_rad, 'zdr': zdr_linear
    }

def estimate_zdr_zh(signal_h, signal_v):
    """从信号估计ZDR"""
    power_h = np.mean(np.abs(signal_h)**2)
    power_v = np.mean(np.abs(signal_v)**2)
    zdr_db = 10 * np.log10(power_h / power_v + 1e-10)
    return zdr_db

def estimate_phi_dp_hshv(signal_h, signal_v, prf):
    """从HSHV信号估计差分相位"""
    # 使用交叉相关估计相位差
    n_h = len(signal_h)
    n_v = len(signal_v)
    n = min(n_h, n_v)
    
    # 计算复数相关系数
    cross_corr = np.mean(signal_h[:n] * np.conj(signal_v[:n]))
    phi_dp = np.angle(cross_corr)
    return phi_dp

## 2. SHV 与 HSHV 信号时序对比

In [ ]:
def plot_mode_comparison(velocity=20.0, wavelength=0.10, prf=1000, n_pulses=64,
                         differential_reflectivity=2.0, differential_phase=30.0):
    """对比SHV和HSHV模式的信号"""
    
    data = simulate_shv_hshv(velocity, wavelength, prf, n_pulses, 
                              differential_reflectivity, differential_phase)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # ===== SHV 模式 =====
    ax1 = axes[0, 0]
    t = data['shv']['t'] * 1000  # ms
    ax1.plot(t, np.real(data['shv']['Zh']), 'b-', label='Zh 实部', linewidth=1.5)
    ax1.plot(t, np.imag(data['shv']['Zh']), 'b--', label='Zh 虚部', linewidth=1.5)
    ax1.plot(t, np.real(data['shv']['Zv']), 'r-', label='Zv 实部', linewidth=1.5)
    ax1.plot(t, np.imag(data['shv']['Zv']), 'r--', label='Zv 虚部', linewidth=1.5)
    ax1.set_xlabel('时间 (ms)', fontsize=10)
    ax1.set_ylabel('幅度', fontsize=10)
    ax1.set_title(f'SHV模式 - H和V通道同时采样 (采样率={prf}Hz)', fontsize=11)
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # ===== HSHV 模式 =====
    ax2 = axes[0, 1]
    t_h = data['hshv']['t_h'] * 1000
    t_v = data['hshv']['t_v'] * 1000
    ax2.plot(t_h, np.real(data['hshv']['Zh']), 'bs-', label='H脉冲', markersize=5)
    ax2.plot(t_v, np.real(data['hshv']['Zv']), 'ro-', label='V脉冲', markersize=5)
    ax2.set_xlabel('时间 (ms)', fontsize=10)
    ax2.set_ylabel('信号实部', fontsize=10)
    ax2.set_title(f'HSHV模式 - H和V交替采样 (采样率={prf/2:.0f}Hz)', fontsize=11)
    ax2.legend(loc='upper right', fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # ===== 相位对比 =====
    ax3 = axes[1, 0]
    phase_shv = np.angle(data['shv']['Zh']) - np.angle(data['shv']['Zv'])
    phase_shv = np.unwrap(phase_shv)
    ax3.plot(t, phase_shv, 'g-', linewidth=2, label='SHV相位移')
    ax3.axhline(y=data['phi_dp'], color='red', linestyle='--', 
                label=f'真实ΦDP={differential_phase:.1f}°')
    ax3.set_xlabel('时间 (ms)', fontsize=10)
    ax3.set_ylabel('相位差 (rad)', fontsize=10)
    ax3.set_title('SHV模式: H-V相位差随时间变化', fontsize=11)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # ===== ZDR估计对比 =====
    ax4 = axes[1, 1]
    # SHV模式估计
    zdr_shv = estimate_zdr_zh(data['shv']['Zh'], data['shv']['Zv'])
    
    # HSHV模式估计
    zdr_hshv = estimate_zdr_zh(data['hshv']['Zh'], data['hshv']['Zv'])
    
    modes = ['SHV', 'HSHV']
    zdr_values = [zdr_shv, zdr_hshv]
    colors = ['steelblue', 'coral']
    bars = ax4.bar(modes, zdr_values, color=colors, alpha=0.7, edgecolor='black')
    ax4.axhline(y=differential_reflectivity, color='green', linestyle='--', 
                linewidth=2, label=f'真实ZDR={differential_reflectivity:.1f}dB')
    ax4.set_ylabel('估计 ZDR (dB)', fontsize=10)
    ax4.set_title('两种模式的ZDR估计对比', fontsize=11)
    ax4.legend()
    ax4.set_ylim(0, 5)
    for bar, val in zip(bars, zdr_values):
        ax4.text(bar.get_x() + bar.get_width()/2, val + 0.1, f'{val:.2f}dB', 
                ha='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # 打印统计
    print(f"\n=== 模式对比统计 ===")
    print(f"设定ZDR: {differential_reflectivity:.1f} dB")
    print(f"SHV估计ZDR: {zdr_shv:.2f} dB")
    print(f"HSHV估计ZDR: {zdr_hshv:.2f} dB")
    print(f"\n时间采样率:")
    print(f"  SHV模式: {prf} Hz (H和V同时)")
    print(f"  HSHV模式: {prf/2:.0f} Hz (H和V交替)")

In [ ]:
interact_mode = interact(plot_mode_comparison,
    velocity=widgets.FloatSlider(min=0, max=50, step=1, value=20.0, 
                               description='径向速度 (m/s)'),
    wavelength=widgets.Dropdown(options=[(0.032, 'X-band'), (0.054, 'C-band'), (0.107, 'S-band')],
                                value=0.107, description='波段'),
    prf=widgets.FloatSlider(min=500, max=2000, step=100, value=1000, description='PRF (Hz)'),
    n_pulses=widgets.IntSlider(min=32, max=128, step=16, value=64, description='脉冲数'),
    differential_reflectivity=widgets.FloatSlider(min=0, max=5, step=0.1, value=2.0, 
                                                  description='ZDR (dB)'),
    differential_phase=widgets.FloatSlider(min=0, max=180, step=5, value=30.0, 
                                           description='ΦDP (度)')
)

## 3. 模式选择指南

In [ ]:
def print_mode_comparison():
    """打印模式对比表格"""
    print("=" * 70)
    print("SHV vs HSHV 模式对比")
    print("=" * 70)
    print(f"{'特性':<25} {'SHV模式':<20} {'HSHV模式':<20}")
    print("-" * 70)
    print(f"{'采样率':<25} {'完整PRF':<20} {'PRF/2':<20}")
    print(f"{'发射功率分配':<25} {'H和V各一半':<20} {'单通道全功率':<20}")
    print(f"{'灵敏度':<25} {'较低 (-3dB)':<20} {'较高':<20}")
    print(f"{'差分相位估计':<25} {'脉冲内直接估计':<20} {'需交叉估计':<20}")
    print(f"{'速度估计精度':<25} {'相同PRF下更高':<20} {'PRF减半稍低':<20}")
    print(f"{'适用场景':<25} {'快速变化降水':<20} {'远距离探测':<20}")
    print("=" * 70)

In [ ]:
print_mode_comparison()

## 练习题

1. **灵敏度计算**：SHV模式由于功率分路，灵敏度降低多少dB？

2. **采样率影响**：当PRF=1000Hz时，HSHV模式的有效采样率是多少？这会影响什么？

3. **相位估计**：在HSHV模式下，如何估计H和V通道之间的相位差？

4. **模式选择**：对于200km外的弱降水，应选择哪种模式？为什么？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 2, Springer.